# पाठ 18 (अनुवर्ती): रसीदें जो प्रमाणित करती हैं कि एक *मानव* ने क्रिया को अधिकृत किया

यह पाठ प्रमाणित करता है कि **एजेंट** ने क्या किया और **गेट** ने क्या निर्णय लिया। यह नोटबुक वह गायब आधा जोड़ता है: प्रमाण कि एक **नामित मानव** ने **सटीक** क्रिया को मंजूरी दी — एक अलग, मानव-के पास हस्ताक्षर जो पूरी कैनोनिकल क्रिया पर होता है, ऑफ़लाइन सत्यापित।

यहां दोनों कलाकृतियां उपयोग करती हैं **उसी लिफाफे की आकृति जिसका इस्तेमाल पाठ की रसीदों में हुआ**: एक फ्लैट पेलोड जिसमें `type` फील्ड होता है, Ed25519 द्वारा कैनोनिकल पेलोड बाइट्स के SHA-256 पर हस्ताक्षर किया गया, जिसमें एक संरचित `signature` ऑब्जेक्ट जुड़ा होता है (और हस्ताक्षरित बाइट्स से बाहर रखा गया)। अनुमोदन रसीद एक नया `type` (`human.approval.v1`) है जो क्रिया प्रकार के साथ होती है, इसलिए एक `verify_chain` मुख्य नोटबुक में बनाए गए समान कोड पथ के साथ दोनों कलाकृतियों को कवर करता है। यह इंटरनेट-ड्राफ्ट में सह-हस्ताक्षर मार्ग की आकृति भी है जिस पर यह पाठ आधारित है (draft-farley-acta-signed-receipts)।

मुख्य नोटबुक के डेमो वेरिफायर की तुलना में एक जानबूझकर उन्नयन: यहां वेरिफायर `signature.key_id` को एक **पिन किए गए कुंजी रजिस्ट्री** के विरुद्ध हल करता है बजाय उस सार्वजनिक कुंजी पर भरोसा करने के जो रसीद के अंदर होती है। यह वह उत्पादन दशा है जिसे पाठ की अपनी चेकलिस्ट अनुशंसा करती है ("सत्यापन सार्वजनिक कुंजी प्रकाशित करें"), और यही वह कारण है जो किजोड़ारी को एक अस्वीकार के रूप में बनाता है न कि अपनी-अपनी कुंजी का बाइपास।

इस नोटबुक का नियम है: **हस्ताक्षरित अनुमोदन अपने आप में अधिकार नहीं है।** अधिकार केवल तभी मौजूद होता है जब अनुमोदन रसीद और क्रिया रसीद निष्पादन समय पर अभी भी उसी कैनोनिकल क्रिया से बंधी हों, एक नीति संस्करण, कुंजी, और समाप्ति के अंतर्गत जो अभी भी वर्तमान हों, और एक ऐसा अनुमोदन जो पहले से उपयोग नहीं किया गया हो। हर विफलता एक **अलग कारण** के साथ अस्वीकार करती है, ताकि आप यह बता सकें कि *अधिकार पुराना हो गया* या *निष्पादित क्रिया बदल गई*।


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## सटीक क्रिया

अनुमोदन की इकाई **कैनॉनिकल क्रिया वस्तु** है — "रिफंड अनुमोदित करें" जैसे अस्पष्ट लेबल नहीं, बल्कि सटीक, पूर्ण रूप से निर्दिष्ट क्रिया। पूरे वस्तु पर हस्ताक्षर करना (और उससे एक डाइजेस्ट निकालना) हमें बाद में साबित करने देता है कि मानव ने *यह* मंजूर किया और कुछ नहीं।


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## एक लिफाफा, दो प्राधिकरण

प्रत्येक रसीद पाठ का लिफाफा होती है: एक फ्लैट पेलोड जिसमें एक `type` फ़ील्ड होता है, साथ ही एक `signature` ऑब्जेक्ट (`alg`, `sig`, `key_id`) जो साइन किए गए बाइट्स का हिस्सा **नहीं** होता। `verify_envelope` दोनों प्रकार की रसीदों के लिए साझा संरचनात्मक + सिग्नेचर चेक है; जो **पिन्ड की रजिस्ट्री** यह `signature.key_id` के खिलाफ हल करता है, वही प्राधिकरणों को अलग रखता है:

- **स्वीकृति रसीद** (`human.approval.v1`) — नामित अनुमोदक, पूरा कैनोनिकल क्रिया **और उसका डिजेस्ट**, `policy_version`, जारी करने और समाप्ति के टाइमस्टैम्प। एक बार उपभोग को चेन स्तर पर ट्रैक किया जाता है।
- **क्रिया रसीद** (`agent.action.v1`) — एजेंट पहचान, `run_id`, वही कैनोनिकल क्रिया **डिजेस्ट**, निष्पादन परिणाम + टाइमस्टैम्प, और `parent_approval_ref`: अनुमोदन का `receipt_hash`, वही कन्वेंशन जैसा कि पाठ की चेन में `previous_receipt_hash` होता है।

साझा `action_digest` फ़ील्ड वह जोड़ी है जिस पर बाइंडिंग निर्भर करती है। `key_id` केवल लुकअप संकेत के रूप में सिग्नेचर ऑब्जेक्ट में रहता है: इसे किसी अलग पिन्ड की की ओर इंगित करने से सिग्नेचर चेक विफल हो जाता है, इसलिए इसका कोई लाभ नहीं होता।


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: जहां वास्तव में बंधन तय होता है

`verify_chain` दो हस्ताक्षर जांचों का एक सहूलियत आवरण **नहीं** है। यह एक स्थान है जहाँ साझा मानक `action_digest`, नीति/कुंजी/समाप्ति की **ताजगी** की मंजूरी, और मंजूरी की **एकबारगी खपत** एक साथ जांची जाती हैं, उस क्रिया के खिलाफ जो इस समय *अमल में लाई जा रही है*।

प्रत्येक अस्वीकार एक **विशिष्ट कारण** के साथ अस्वीकृत करता है, ताकि अस्वीकृति का पाठक यह बता सके कि प्राधिकरण समयाभावित हो गया है (नीति बदली गई, कुंजी बदली गई, मंजूरी समाप्त हो गई, मंजूरी खपत हो गई) या अभी भी मान्य मंजूरी के तहत निष्पादित क्रिया में बदलाव हुआ है (डाइजेस्ट प्रतिस्थापन)।


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## बाइंडिंग क्या पकड़ता है

नीचे हर मामले में **एक अलग कारण** के साथ **बंद** हो जाता है। पहला ब्लॉक क्लासिक सेट है (टैम्पर, भ्रमित डेप्यूटी, रिप्ले, किसी भी अधिकार पर फर्जी और गलत रूप से बनाई गई इनपुट)। दूसरा ब्लॉक वह जो संपत्ति को सत्यापित किए जाने के बजाय वास्तविक बनाता है:

- **पुराना अधिकार** — हस्ताक्षर अभी भी मान्य है, लेकिन नीति संस्करण बदल गया, अनुमोदक कुंजी पिन की गई रजिस्ट्री से हटाई गई, या अनुमोदन निष्पादन से पहले समाप्त हो गया;
- **डाइजेस्ट प्रतिस्थापन** — एक वैध रूप से हस्ताक्षरित क्रिया रसीद जिसकी `parent_approval_ref` एक *वास्तविक* अनुमोदन की ओर इशारा करती है, लेकिन उस अनुमोदन का मानक क्रिया डाइजेस्ट उस क्रिया से मेल नहीं खाता जो वास्तव में निष्पादित की जा रही है।


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## यह क्या साबित करता है — और क्या नहीं

**साबित करता है:** एक नामित व्यक्ति ने *इस सटीक कैनोनिकल क्रिया* (पूर्ण क्रिया + डाइजेस्ट, पिन्ड रजिस्ट्री से हल हुई कुंजी से हस्ताक्षरित) को मंजूरी दी, और एजेंट ने *ठीक वही मंजूर की गई क्रिया* (वही डाइजेस्ट, स्वीकृति `receipt_hash` द्वारा अनुमोदन से बंधी हुई, पाठ का अपना चेन कन्वेंशन) निष्पादित की — जबकि अनुमोदन की नीति संस्करण, कुंजी, और समाप्ति अभी भी वर्तमान थे, बिल्कुल एक बार। यदि किसी भी पक्ष में बदलाव होता है, तो चेइन बंद हो जाती है, और अस्वीकार करने का कारण आपको बताता है कि **कौन** सी संपत्ति टूटी: पुरानी प्राधिकरण बनाम बदली हुई क्रिया।

**साबित नहीं करता:** कि अनुमोदन UI ने व्यक्ति को वही दिखाया जो वे हस्ताक्षर करने का सोच रहे थे (WYSIWYS अपनी समस्या है), कि कुंजी को रोटेशन से पहले जबरन या चुराया नहीं गया था, या कि डाउनस्ट्रीम प्रभाव क्रिया से मेल खाते थे। हस्ताक्षरित ≠ प्राधिकृत: पुरानी नीति पर वैध हस्ताक्षर, परिवर्तित कुंजी, समाप्त हो चुका समय, या एक अलग डाइजेस्ट यहाँ कुछ भी नहीं देता है।

दोनों रसीद प्रकार पाठ के लिफाफे और एक `verify_chain` कोड पथ को जानबूझकर साझा करते हैं: आप जो बाइंडिंग मुख्य नोटबुक में क्रिया रसीदों के लिए बनाते हैं वही कोड है जो व्यक्ति की अनुमोदन जांचता है। एक सत्यापन अनुबंध, अलग-अलग पिन्ड प्राधिकरण, केवल सगाई कैनोनिकल क्रिया डाइजेस्ट द्वारा और कुछ नहीं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
